<a href="https://colab.research.google.com/github/stanley-yang-2001/AnyLang-MP-warmup/blob/main/Reproduction_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div class="markdown-google-sans">

## Introduction
The goal of this Colab is to reproduce the result presented in the paper *Addressing wearable sleep tracking inequity: a new dataset and novel methods for a population with sleep disorders*. The codebase for the reproduction can be found in the github using this [link](https://github.com/WillKeWang/DREAMT_FE).

##  Setup

The instruction for the setup of the reproduction can be found in the readme file in the github for the codebase. First thing first, we clone the repository.




In [ ]:
# clone the remote repository, the ! tells the Colab to run this command in shell environment
!git clone https://github.com/WillKeWang/DREAMT_FE.git

Cloning into 'DREAMT_FE'...
remote: Enumerating objects: 230, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 230 (delta 9), reused 6 (delta 6), pack-reused 212 (from 1)
Receiving objects: 100% (230/230), 99.75 MiB | 7.17 MiB/s, done.
Resolving deltas: 100% (9/9), done.
Updating files: 100% (213/213), done.


In [ ]:
# Check if the repo exist
import os

folder = "DREAMT_FE"

if os.path.exists(folder):
    print(f"The folder '{folder}' exists.")
else:
    print(f"The folder '{folder}' does not exist.")

The folder 'DREAMT_FE' exists.


After we clone the repository, we'll need to setup the Conda environment from the .yml file. First, install the Conda library.

In [ ]:
!pip install -q condacolab

Then, set up the conda environment. Note that this will restart the Python kernel.

In [ ]:
import condacolab
# if the environment is already installed, this line will print a message instead
condacolab.install()


⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:25
🔁 Restarting kernel...


Lastly, create the environment using the environment.yml file. This will take a while.

In [1]:
import os
os.chdir('/content/DREAMT_FE/')
# removing and creating environment again just in case
!conda env remove -n dreamt -y

!conda env create -f environment.yml

FileNotFoundError: [Errno 2] No such file or directory: '/content/DREAMT_FE/'

Now, to run a file in conda environment, use the following format:

- `!conda run -n dreamt command`.

replace the `command` with whatever command you want to run.

Now running the main.py file.

In [ ]:
import os
os.chdir('/content/DREAMT_FE/')
!conda run -n dreamt python -u main.py 2>&1 | tee output.log
print("finished")

MemTotal:       13286964 kB
MemFree:         8407548 kB
MemAvailable:   11981216 kB

clean df
Index([    1,     2,     3,     4,     5,     6,     7,     8,     9,    10,
       ...
       85061, 85062, 85063, 85064, 85065, 85066, 85067, 85068, 85069, 85070],
      dtype='int64', length=82671)

new_features df
<built-in method index of list object at 0x78565ad69700>

good_quality_sids df
<built-in method index of list object at 0x78565ae344c0>

finished
MemTotal:       13286964 kB
MemFree:         9988736 kB
MemAvailable:   12149280 kB


## Extension

Below is the pipeline for the focal loss extension of the project. To run the extension, please copy the following file to the main directory along with main.py, main_cv.py, etc. Name it focalloss.py

In [ ]:
# experiment_focalloss.py
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, accuracy_score, cohen_kappa_score
from sklearn.model_selection import KFold

from datasets import *
from utils import *
from models import *  # uses LightGBM_engine, LSTM_engine, LSTM_eval, etc.

# ==============================================================
# 1️⃣ Data Preparation
# ==============================================================
threshold = 0.2
quality_df_dir = "results/quality_scores_per_subject.csv"
features_dir = "dataset_sample/features_df/"
info_dir = "dataset_sample/participant_info.csv"

clean_df, new_features, good_quality_sids = data_preparation(
    threshold, quality_df_dir, features_dir, info_dir
)
SW_df, final_features = split_data(clean_df, good_quality_sids, new_features)
group_variable = ["AHI_Severity"]

kf = KFold(n_splits=5, shuffle=True, random_state=42)
all_results = []

for fold, (train_idx, test_idx) in enumerate(kf.split(good_quality_sids), 1):
    print(f"\n=== Fold {fold} ===")

    train_sids = [good_quality_sids[i] for i in train_idx[:-8]]
    val_sids = [good_quality_sids[i] for i in train_idx[-8:]]
    test_sids = [good_quality_sids[i] for i in test_idx]

    # Split and resample
    X_train, y_train, group_train = train_test_split(SW_df, train_sids, final_features, group_variable)
    X_val, y_val, group_val = train_test_split(SW_df, val_sids, final_features, group_variable)
    X_test, y_test, group_test = train_test_split(SW_df, test_sids, final_features, group_variable)
    X_train_resampled, y_train_resampled, group_train_resampled = resample_data(X_train, y_train, group_train, group_variable)

    # ==============================================================
    # 2️⃣ Base LightGBM (unchanged)
    # ==============================================================
    print("[STEP] Training LightGBM base model...")
    final_lgb_model = LightGBM_engine(X_train_resampled, y_train_resampled, X_val, y_val)

    prob_ls_train, len_train, true_ls_train = compute_probabilities(
        train_sids, SW_df, final_features, "lgb", final_lgb_model, group_variable)
    prob_ls_test, len_test, true_ls_test = compute_probabilities(
        test_sids, SW_df, final_features, "lgb", final_lgb_model, group_variable)

    # ==============================================================
    # 3️⃣ LSTM Baseline (CrossEntropyLoss)
    # ==============================================================
    print("[STEP] Training baseline LSTM (CrossEntropyLoss)...")
    dataloader_train = LSTM_dataloader(prob_ls_train, len_train, true_ls_train, batch_size=32)
    dataloader_test = LSTM_dataloader(prob_ls_test, len_test, true_ls_test, batch_size=1)

    LSTM_model_ce = LSTM_engine(
        dataloader_train,
        num_epoch=10,
        hidden_layer_size=32,
        learning_rate=0.001
    )
    lstm_ce_results_df = LSTM_eval(LSTM_model_ce, dataloader_test, true_ls_test, "LSTM_CE")

    # Compute metrics
    # Directly collect the aggregated metrics from the eval outputs
    metrics_ce = lstm_ce_results_df.iloc[0].to_dict()
    #print("\n=== LSTM Loss Function Comparison ===")
    #print(results_df.to_string(index=False))


    # ==============================================================
    # 4️⃣ LSTM with Focal Loss (Extension)
    # ==============================================================
    print("[STEP] Training LSTM with Focal Loss...")

    class FocalLoss(nn.Module):
      def __init__(self, alpha=1, gamma=2):
          super().__init__()
          self.alpha = alpha
          self.gamma = gamma
          self.bce = nn.BCELoss(reduction="none")

      def forward(self, inputs, targets):
          # Convert logits -> probability for positive class
          probs = torch.softmax(inputs, dim=1)[:, 1]  # take P(class=1)
          targets = targets.float()
          bce_loss = self.bce(probs, targets)
          pt = torch.exp(-bce_loss)
          loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
          return loss.mean()

    dataloader_train = LSTM_dataloader(prob_ls_train, len_train, true_ls_train, batch_size=32)
    dataloader_test = LSTM_dataloader(prob_ls_test, len_test, true_ls_test, batch_size=1)
    loss_fn = FocalLoss(alpha=1, gamma=2)
    LSTM_model_focal = LSTM_engine(
        dataloader_train,
        num_epoch=10,
        hidden_layer_size=32,
        learning_rate=0.001,
        loss_fn = loss_fn
    )

    # Replace the loss manually if desired inside LSTM_engine OR retrain manually
    # Here we re-train briefly with focal loss on top
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = LSTM_model_focal.to(device)
    loss_fn = FocalLoss(alpha=1, gamma=2)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    '''
    for epoch in range(3):  # quick fine-tune
        model.train()
        total_loss = 0
        for batch in dataloader_train:
            sample = batch["sample"].to(device)
            label = batch["label"].to(device)
            length = batch["length"]
            optimizer.zero_grad()
            pred = model(sample, length)
            loss = loss_fn(pred, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1} Focal fine-tune — Loss: {total_loss/len(dataloader_train):.4f}")
    '''
    # Evaluate
    lstm_focal_results_df = LSTM_eval(model, dataloader_test, true_ls_test, "LSTM_Focal")

    #y_true_focal = np.concatenate(true_ls_test)
    #y_pred_prob_focal = lstm_focal_results_df["Pred_Prob"].values
    #y_pred_focal = (y_pred_prob_focal > 0.5).astype(int)
    # Directly collect the aggregated metrics from the eval outputs
    metrics_focal = lstm_focal_results_df.iloc[0].to_dict()

    # results_df_focal = pd.DataFrame([metrics_focal_loss, metrics_focal])
    print("\n=== LSTM Loss Function Comparison ===")
    results_df = pd.DataFrame([metrics_ce, metrics_focal])
    print(results_df.to_string(index=False))


    all_results.extend([metrics_ce, metrics_focal])

# ==============================================================
# 5️⃣ Summary
# ==============================================================
df_results = pd.DataFrame(all_results)
print("\n=== LSTM Baseline vs Focal Loss Comparison ===")
print(df_results.groupby("Model").mean().round(4))
df_results.to_csv("results/lstm_focalloss_comparison.csv", index=False)


In [ ]:
import os, time
os.chdir('/content/DREAMT_FE/')
start = time.time()
!time conda run -n dreamt python focalloss.py
end = time.time()
print(f"Elapsed time: {end - start:.2f} seconds")